<a id="top"></a>

# Demo: the description is the tool

```yaml
title: "Demo: the description is the tool"
keywords: tool description, recruitment ad, contract, sparse vs rich vs misleading, selection step, haiku contrast, StructuredTool, langchain, teacher demo
estimated duration: 14
```

> **Lesson:** L08. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Behavior is **distributional**: run each variant twice so the class sees variance, not a single result. Clear outputs before committing.
>
> **Model-agnostic by design.** The model is a LangChain `ChatAnthropic`; each tool variant is built with `StructuredTool.from_function`, which keeps the **same** function and inferred argument schema and swaps only the description prose — exactly the variable this demo isolates. The key loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: **same name, same schema, different description → different behavior.** The description is the model's only training signal at inference time for *when* to reach for the tool. We also re-run on **Haiku 4.5** to show the design gap widens on a smaller model.

## Contents

- [1. Setup — one tool, three descriptions](#1-setup--one-tool-three-descriptions)
- [2. Read the three descriptions side by side](#2-read-the-three-descriptions-side-by-side)
- [3. The straightforward prompt (P1)](#3-the-straightforward-prompt-p1)
- [4. The ambiguous prompt (P2) — where the description earns its keep](#4-the-ambiguous-prompt-p2--where-the-description-earns-its-keep)
- [5. The misleading description's delayed cost](#5-the-misleading-descriptions-delayed-cost)
- [6. The gap widens on a smaller model (Haiku)](#6-the-gap-widens-on-a-smaller-model-haiku)
- [7. Takeaways](#7-takeaways)

## 1. Setup — one tool, three descriptions

One implementation: `lookup_user(query)` over a tiny in-memory user table. Three tool *variants* built with `StructuredTool.from_function` that differ **only** in the `description` — sparse, rich, misleading — via a `make_tool(description)` helper. Two models: Sonnet 4.6 (anchor) and Haiku 4.5 (the smaller-model contrast). Live cells are gated on `LIVE`.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model — the design gap bites harder
LIVE = bool(get_settings().anthropic_api_key)


# The implementation NEVER changes across the three variants — only the description does.
USERS: dict[str, dict[str, str]] = {
    "alex@example.com": {"name": "Alex Kim", "email": "alex@example.com"},
    "alex_kim": {"name": "Alex Kim", "email": "alex@example.com"},
    "priya@example.com": {"name": "Priya Rao", "email": "priya@example.com"},
}


def lookup_user(query: str) -> dict[str, str]:
    """Return the user record for an email or username, or {} if not found."""
    return USERS.get(query.strip().lower(), {})


SPARSE = "Looks up a user."
RICH = (
    "Look up a single user record by exact email or username. "
    "Accepts one query string: an email (e.g. 'alex@example.com') or a username "
    "(e.g. 'alex_kim'). Returns {'name', 'email'} for that one user, or {} if no exact "
    "match. Do NOT call this to list or summarize users in general, and do not guess a "
    "query — if the user has not named a specific person, ask them who they mean instead."
)
MISLEADING = (
    "Looks up any information about any user — email, billing, preferences, and full "
    "account history. Call this for anything user-related."
)  # the implementation returns only {name, email}; this OVERSTATES it.


def make_tool(description: str) -> StructuredTool:
    """Build the lookup_user tool with a swappable description (name + arg schema fixed).

    StructuredTool.from_function keeps the SAME function and inferred argument
    schema and swaps only the description prose — exactly the variable this demo
    isolates. bind_tools accepts the returned tool like any other.
    """
    return StructuredTool.from_function(
        func=lookup_user, name="lookup_user", description=description
    )


def show_turn(reply: AIMessage) -> None:
    """Print a reply's text and any tool calls (name + args)."""
    if reply.text:
        print(f"  text       {reply.text!r}")
    for call in reply.tool_calls:
        args = ", ".join(f"{k}={v!r}" for k, v in call["args"].items())
        print(f"  tool call  {call['name']}({args})")


P1 = "Find me the email for the user named Alex."  # straightforward
P2 = "Tell me about our users."  # ambiguous — the revealing case

if LIVE:
    key = require_anthropic_key()
    sonnet = ChatAnthropic(model=SONNET, api_key=key, max_tokens=300)
    haiku = ChatAnthropic(model=HAIKU, api_key=key, max_tokens=300)

print("setup ready (LIVE =", LIVE, ")")

[↑ Back to top](#top)

## 2. Read the three descriptions side by side

Before any model call, put the three descriptions on screen and read them aloud. Ask the class (rhetorically) to predict which produces the best tool use. The name and schema are **identical** in all three — only this prose changes.

In [ ]:
for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
    print(f"--- {label} ---")
    print(" ", desc)
    print()

[↑ Back to top](#top)

## 3. The straightforward prompt (P1)

Run P1 against each description. All three usually produce a tool call — but watch the **arguments**: the rich description tends to make the model format the query more precisely (it copies the example format), while sparse leaves it guessing the shape.

In [ ]:
if LIVE:
    for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
        reply = sonnet.bind_tools([make_tool(desc)]).invoke([HumanMessage(P1)])
        print(f"=== P1 / {label} ===")
        show_turn(reply)
        print()
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. The ambiguous prompt (P2) — where the description earns its keep

Run the **ambiguous** P2 against each description. This is where behavior diverges: **sparse** often calls the tool with a garbage query; **rich** usually asks for clarification or refuses (its description says *don't guess*); **misleading** calls the tool expecting rich data. Run each twice — the lesson is the *distribution*, not one run.

In [ ]:
if LIVE:
    for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
        print(f"=== P2 / {label} (two runs) ===")
        bound = sonnet.bind_tools([make_tool(desc)])
        for _ in range(2):
            show_turn(bound.invoke([HumanMessage(P2)]))
            print()
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 5. The misleading description's delayed cost

The misleading description's damage shows up on the *follow-up*. Run P1 with it, then ask for *billing history* — data the tool never returns. The model often calls the same tool again expecting more, because the description **promised** it. A description that overstates the tool is as harmful as one that says too little.

In [ ]:
if LIVE:
    bound = sonnet.bind_tools([make_tool(MISLEADING)])
    messages = [HumanMessage(P1)]
    first = bound.invoke(messages)
    print("=== first turn (P1, misleading) ===")
    show_turn(first)

    # Feed back a thin tool result, then ask for data the tool can't provide.
    if first.tool_calls:
        call = first.tool_calls[0]
        messages.append(first)
        messages.append(
            ToolMessage(content=str(lookup_user(**call["args"])), tool_call_id=call["id"])
        )
        messages.append(HumanMessage("Now show me their billing history."))
        followup = bound.invoke(messages)
        print("\n=== follow-up: 'show me their billing history' ===")
        show_turn(followup)
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 6. The gap widens on a smaller model (Haiku)

Re-run the revealing P2 case on **Haiku 4.5**. A smaller model leans harder on the description, so the sparse-vs-rich gap is sharper: sparse Haiku is more likely to fire a garbage call, rich Haiku more likely to follow the *don't guess* instruction. The design error bites harder where the model has less to fall back on.

In [ ]:
if LIVE:
    for label, desc in [("SPARSE", SPARSE), ("RICH", RICH)]:
        reply = haiku.bind_tools([make_tool(desc)]).invoke([HumanMessage(P2)])
        print(f"=== P2 / {label} / HAIKU ===")
        show_turn(reply)
        print()
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 7. Takeaways

- **Same tool, same model, same task — only the description changed, and behavior changed dramatically.** The description is the model's only training signal at inference time for *when* to reach for the tool.
- A description is two things at once: a **recruitment ad** (when to reach for it) and a **contract** (what it returns). A mismatch in either dimension cascades into bad calls.
- **Write for the model's selection step, not the human reader.** Code comments are for humans; tool descriptions are for the model.
- An **overstated** description (misleading) is as harmful as a sparse one — match the description to the implementation exactly.
- Descriptions reduce variance, they don't eliminate it — which is why the **schema** ([L0807](L0807_lecture.ipynb)) is the next line of defense. The [L0806 lab](L0806_lab_empty.ipynb) has you rewrite a sparse description into a rich one.

[↑ Back to top](#top)